<a href="https://colab.research.google.com/github/Adeyemi-Joshua/data-science-cohort/blob/main/FPL_Data_Science_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fantasy Premier League: Who Deserves More Points?

A full data science project, start to finish, on live data from the official Fantasy Premier League API.

**The question:** some players score fewer FPL points than their performances deserve. Bad luck, wasted chances, a keeper having a blinder. If we can spot them, we can buy them before their points catch up.

**The plan**

1. Ask the question
2. Get the data
3. Understand the data
4. Clean the data
5. Explore the data
6. Build a model
7. Evaluate the model
8. Use the model
9. Share the work


## 1. Ask the question

- **What** are we trying to find? Players whose points are lower than their underlying numbers say they should be.
- **Why** does it matter? In FPL you have a fixed budget. Buying a player just before his points go up is how you win your mini league.
- **How** will we answer it? Teach a model what points a player "should" have from how well he actually played (chances, minutes, defending), then compare that to the points he really got.


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 2. Get the data

The FPL website gets its data from a public API. We ask it for data and it sends back JSON, the same thing the website uses to show you player prices and points.


In [ ]:
url = "https://fantasy.premierleague.com/api/bootstrap-static/"
data = requests.get(url).json()

data.keys()

Three parts are useful to us:

- `elements`: one row per player
- `teams`: the 20 Premier League clubs
- `element_types`: the four positions (GKP, DEF, MID, FWD)


In [ ]:
players = pd.DataFrame(data["elements"])
teams = pd.DataFrame(data["teams"])
positions = pd.DataFrame(data["element_types"])

players.shape

## 3. Understand the data

Hundreds of players and around 100 columns. Let's look before we touch anything.


In [ ]:
players.head()

Things to notice:

- `team` and `element_type` are **numbers**, not names. We need to translate them.
- `now_cost` is in tenths: 105 means £10.5m.
- Columns like `expected_goals` look like numbers but are stored as **text**.

We only need a handful of columns for our question.


In [ ]:
columns = [
    "web_name",
    "team",
    "element_type",
    "now_cost",
    "status",
    "minutes",
    "starts",
    "goals_scored",
    "assists",
    "clean_sheets",
    "expected_goals",
    "expected_assists",
    "expected_goals_conceded",
    "defensive_contribution",
    "selected_by_percent",
    "total_points",
]

df = players[columns].copy()
df.dtypes

## 4. Clean the data

Real data is never ready to use. Here are the fixes:


In [ ]:
df["team"] = df["team"].map(teams.set_index("id")["name"])
df["position"] = df["element_type"].map(positions.set_index("id")["singular_name_short"])
df["price"] = df["now_cost"] / 10

text_numbers = [
    "expected_goals",
    "expected_assists",
    "expected_goals_conceded",
    "selected_by_percent",
]
df[text_numbers] = df[text_numbers].apply(pd.to_numeric)

df = df.drop(columns=["element_type", "now_cost"])
df.head()

Any missing values?


In [ ]:
df.isna().sum().sum()

A player who has played 20 minutes all season tells us nothing. We keep only players with at least **270 minutes** (three full matches).


In [ ]:
df = df[df["minutes"] >= 270].reset_index(drop=True)
df.shape

## 5. Explore the data

Before any machine learning, we look. Charts answer simple questions fast.

**Who has the most points so far?**


In [ ]:
top10 = df.nlargest(10, "total_points")

sns.barplot(data=top10, x="total_points", y="web_name", hue="position", dodge=False)
plt.title("Top 10 FPL scorers")
plt.xlabel("Total points")
plt.ylabel("")
plt.show()

**Do expensive players score more?**


In [ ]:
sns.scatterplot(data=df, x="price", y="total_points", hue="position", alpha=0.7)
plt.title("Price vs points")
plt.xlabel("Price (£m)")
plt.ylabel("Total points")
plt.show()

Mostly yes, but look at the spread. Plenty of cheap players outscore expensive ones. That's where value lives.

**Expected goals vs real goals**

Expected goals (xG) measures the quality of a player's chances. A player above the line is finishing better than expected. A player below it is missing chances, or getting unlucky.


In [ ]:
sns.scatterplot(data=df, x="expected_goals", y="goals_scored", hue="position", alpha=0.7)
plt.axline((0, 0), slope=1, color="grey", linestyle="--")
plt.title("Expected goals vs actual goals")
plt.xlabel("Expected goals (xG)")
plt.ylabel("Goals scored")
plt.show()

**Which positions score the most?**


In [ ]:
sns.boxplot(data=df, x="position", y="total_points", order=["GKP", "DEF", "MID", "FWD"])
plt.title("Points by position")
plt.xlabel("")
plt.ylabel("Total points")
plt.show()

## 6. Build a model

We want the model to learn: _given how a player has played, how many points should he have?_

So the inputs (**features**) are things that describe performance: minutes, chances created, chances taken, defending. We deliberately leave out actual goals and assists, because those are the "luck" part we want to measure.

The thing we predict (the **target**) is total points.


In [ ]:
features = [
    "minutes",
    "starts",
    "price",
    "expected_goals",
    "expected_assists",
    "expected_goals_conceded",
    "defensive_contribution",
]

X = pd.get_dummies(df[features + ["position"]], columns=["position"], dtype=int)
y = df["total_points"]

X.head()

We split the data. The model learns from 80% of players (the **training set**) and we test it on the 20% it has never seen (the **test set**). Like studying past questions, then sitting a real exam.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

len(X_train), len(X_test)

We try two models:

- **Linear regression**: draws the best straight-line relationship between features and points.
- **Random forest**: hundreds of small decision trees that vote. Can capture more complex patterns.


In [ ]:
linear = LinearRegression().fit(X_train, y_train)
forest = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_train, y_train)

## 7. Evaluate the model

Two scores:

- **MAE** (mean absolute error): on average, how many points is the prediction off by? Lower is better.
- **R²**: how much of the difference between players the model explains. 1 is perfect, 0 is useless.

We also compare to a lazy **baseline**: guess the average for everyone. A model that can't beat that is worthless.


In [ ]:
baseline = [y_train.mean()] * len(y_test)

results = pd.DataFrame(
    {
        "MAE": [
            mean_absolute_error(y_test, baseline),
            mean_absolute_error(y_test, linear.predict(X_test)),
            mean_absolute_error(y_test, forest.predict(X_test)),
        ],
        "R²": [
            r2_score(y_test, baseline),
            r2_score(y_test, linear.predict(X_test)),
            r2_score(y_test, forest.predict(X_test)),
        ],
    },
    index=["Guess the average", "Linear regression", "Random forest"],
).round(2)

results

Pick the model with the lower MAE. Let's see how its predictions compare to reality. Points on the line are perfect predictions.


In [ ]:
best_name = results["MAE"].iloc[1:].idxmin()
best_model = {"Linear regression": linear, "Random forest": forest}[best_name]

sns.scatterplot(x=y_test, y=best_model.predict(X_test))
plt.axline((0, 0), slope=1, color="grey", linestyle="--")
plt.title(f"{best_name}: predicted vs actual (test set)")
plt.xlabel("Actual points")
plt.ylabel("Predicted points")
plt.show()

**What does the random forest think matters most?**


In [ ]:
importance = pd.Series(forest.feature_importances_, index=X.columns).sort_values()

importance.plot(kind="barh")
plt.title("What drives FPL points")
plt.xlabel("Importance")
plt.show()

## 8. Use the model

Now we answer the question. Every player gets an **expected points** score. To keep it fair, each player's prediction comes from a model that never saw him during training (`cross_val_predict` handles this).

- `luck` below zero: fewer points than his performances deserve. **Buy candidate.**
- `luck` above zero: more points than he deserves. Could cool off.


In [ ]:
df["expected_points"] = cross_val_predict(best_model, X, y, cv=5).round(1)
df["luck"] = (df["total_points"] - df["expected_points"]).round(1)

show = [
    "web_name",
    "team",
    "position",
    "price",
    "total_points",
    "expected_points",
    "luck",
    "selected_by_percent",
]

**Unluckiest players: due some points**


In [ ]:
df.nsmallest(10, "luck")[show]

**Luckiest players: might cool off**


In [ ]:
df.nlargest(10, "luck")[show]

**Best value: expected points per £1m, available players only**


In [ ]:
df["value"] = (df["expected_points"] / df["price"]).round(1)

(
    df[df["status"] == "a"]
    .sort_values("value", ascending=False)
    .groupby("position")
    .head(3)
    .sort_values(["position", "value"], ascending=[True, False])[show + ["value"]]
)

## 9. Share the work

A finding that stays on your laptop helps nobody. Data scientists share:

- **A short summary** for non-technical people: "These 5 players are underpriced, here's why."
- **The code** on GitHub, so anyone can check it and rerun it.

**Save this notebook to GitHub from Colab:** `File` → `Save a copy in GitHub` → pick your repository → add a commit message → OK.

### Things to remember

- The data is **live**. Run this next week and the answers change.
- Expected points is not a promise. It says who _deserves_ more, not who will get it.
- The model only knows what we gave it. No injuries, no fixtures, no form. Those are ideas for improving it.

### Try it yourself

1. Change the minutes filter from 270 to 450. Do the results change?
2. Add `"clean_sheets"` to the features. Does the MAE get better?
3. Find the best value goalkeeper under £5m.
